# Sampling approach

This notebook draws every sample used in the report. It needs the files written by `1_data_collection_filtering.ipynb` in `data/`.

Our overall question is whether `kubernetes/kubernetes` is healthy, i.e. whether it has healthy practices, including newcomer onboarding, and enough active contributors to fill all relevant roles. The second part can be answered with metrics on the complete data, so there we do not sample. The first part needs a qualitative look at how contributions are actually handled, which requires reading PR discussions by hand; here we sample. We used the three sampling approaches from the lecture, each in a different situation.

In [1]:
import pandas as pd
pd.set_option("display.max_colwidth", 90)

In [2]:
# constants

start = pd.Timestamp("2025-01-01", tz="UTC")
end = pd.Timestamp("2026-01-01", tz="UTC")
seed = 42

In [3]:
# load the output of notebook 1

commits = pd.read_csv("data/commits.csv", parse_dates=["author_date", "commit_date"])
prs = pd.read_csv("data/prs.csv", parse_dates=["merged_at"])
pr_commits = pd.read_csv("data/pr_commits.csv")
person_map = pd.read_csv("data/person_map.csv")
shared = set(pd.read_csv("data/shared_logins.csv").login)

pid = dict(zip(person_map.node, person_map.person_id))
print(len(commits), "commits |", len(prs), "merged PRs |", len(shared), "shared logins")

142253 commits | 58206 merged PRs | 27 shared logins


## No sampling for quantitative metrics

All numbers in this report are computed on the complete data of 2025. The git history has no result limits, and with about 7,000 commits and 2,600 merged PRs it is small enough to process entirely, so sampling would only add error.

In [4]:
c25 = commits[(commits.commit_date >= start) & (commits.commit_date < end)]
p25 = prs[(prs.merged_at >= start) & (prs.merged_at < end)]
print("2025 commits:", len(c25), "| 2025 merged PRs:", len(p25))

2025 commits: 7141 | 2025 merged PRs: 2612


## Random sampling to validate identity merging

To check whether our identity merging is correct, we drew 20 random people that consist of more than one identity and checked each by hand. Any merged person can be wrong, so a simple random sample gives an unbiased estimate of the error rate.

The sample is drawn in notebook 1 (right after the merging, `random_state=42`); here we only load it.

In [5]:
check = pd.read_csv("data/identity_validation_sample.csv")
print(len(check), "people checked by hand, all 20 correct")
check

20 people checked by hand, all 20 correct


,logins,names,emails
0,['narita-1095305'],['Rikiya Narita'],['b2560305@planet.kanazawa-it.ac.jp']
1,['serathius'],['Marek Siarkowicz'],"['marek.siarkowicz@protonmail.com', 'serathius@users.noreply.github.com', 'siarkowicz@..."
2,['devmanishoffl'],['devmanishofficial'],['devmanishofficial@gmail.com']
3,['wojtek-t'],"['Wojciech Tyczynski', 'Wojciech Tyczyński', 'wojtekt']",['wojtekt@google.com']
4,"['atlassian', 'ash2k']",['Mikhail Mazurskiy'],"['126021+ash2k@users.noreply.github.com', 'mmazurskiy@atlassian.com', 'mmazurskiy@gitl..."
5,['aditigupta96'],['Aditi Gupta'],['aditigpta@google.com']
6,['jm-franc'],['Jean-Marc François'],['jmfran@google.com']
7,['bartoszmajsak'],['Bartosz Majsak'],['bartosz.majsak@gmail.com']
8,['shajmakh'],['Shereen Haj'],['shajmakh@redhat.com']
9,['rkolchmeyer'],['Robert Kolchmeyer'],['rkolchmeyer@google.com']


## Author categories (needed for stratification)

The stratified sample groups PRs by the category of their author. The categories are explained and justified in the *Current contributors and involvement* subsection; here we only compute them:
- **Newcomer**: first contribution ever in 2025
- **Frequent**: not new, active in at least 4 months of 2025
- **Returning**: not new, back after at least one year without contributions
- **Occasional**: everyone else

A contribution is a merged PR authored (via the PR login) or a non-merge commit authored (via the commit email). Bots are excluded.

In [6]:
# one row per contribution: (person, date, type)

bots = set(person_map[person_map.is_bot].node)

c = commits[~commits.is_merge].copy()
c["node"] = "email:" + c.author_email
c = c[~c.node.isin(bots)]
act_commits = pd.DataFrame({"person_id": c.node.map(pid), "date": c.commit_date, "type": "commit"})

p = prs.copy()
p["login"] = p.pr_login.str.lower()
p["node"] = "login:" + p.login
p = p[~p.node.isin(bots) & ~p.login.isin(shared)]
act_prs = pd.DataFrame({"person_id": p.node.map(pid), "date": p.merged_at, "type": "pr"})

act = pd.concat([act_commits, act_prs]).dropna()
act = act[act.person_id >= 0]
act["person_id"] = act.person_id.astype(int)
print(len(act), "contributions")

139806 contributions


In [7]:
# categories of everyone active in 2025

first = act.groupby("person_id").date.min()
last_before = act[act.date < start].groupby("person_id").date.max()

y = act[(act.date >= start) & (act.date < end)].copy()
y["month"] = y.date.dt.month
people = y.groupby("person_id").agg(n=("type", "size"), months=("month", "nunique"))
people["first_ever"] = first
people["last_before_2025"] = last_before

def category(r):
    if r.first_ever >= start:
        return "Newcomer"
    if r.months >= 4:
        return "Frequent"
    if r.last_before_2025 < start - pd.Timedelta(days=365):
        return "Returning"
    return "Occasional"

people["category"] = people.apply(category, axis=1)
print(len(people), "people active in 2025")
people.category.value_counts()

553 people active in 2025


category
Newcomer      292
Occasional    105
Frequent       94
Returning      62
Name: count, dtype: int64

In [8]:
# category of the author of every PR merged in 2025

p25 = p25.copy()
p25["person_id"] = ("login:" + p25.pr_login.str.lower()).map(pid)
p25["category"] = p25.person_id.map(people.category)

# lead time: first commit of the PR -> merge
first_commit = pr_commits.merge(commits[["sha", "author_date"]], on="sha").groupby("pr_number").author_date.min()
p25["lead_days"] = (p25.merged_at - p25.pr_number.map(first_commit)).dt.total_seconds() / 86400
p25 = p25.dropna(subset=["category", "lead_days"])
p25 = p25[p25.lead_days >= 0]

share = p25.category.value_counts(normalize=True).round(3)
print("share of 2025 PRs by author category:")
share

share of 2025 PRs by author category:


category
Frequent      0.697
Newcomer      0.190
Occasional    0.080
Returning     0.033
Name: proportion, dtype: float64

Frequent contributors author 70% of all merged PRs. A random sample of 12 PRs would therefore contain only about 2 newcomer PRs, too few to say anything about onboarding. This is why we stratify.

## Stratified sampling for the qualitative analysis

Reading PR discussions by hand does not scale, so we sampled merged 2025 PRs by the category of their author (frequent, occasional and newcomer) and randomly picked four PRs per category with a fixed random seed. A simple random sample would not work here: frequent contributors author 70% of all PRs, so newcomers would hardly appear. For each PR we read the full discussion and noted who responded and how fast, how often the author had to ask for a review, and which actions bots took.

The returning category is left out: it is small (86 PRs) and behaves like the occasional category (see lead times in the next subsection).

In [9]:
strata = ["Frequent", "Occasional", "Newcomer"]
stratified = pd.concat([p25[p25.category == k].sample(4, random_state=seed) for k in strata])
stratified["sample"] = "stratified"
stratified[["category", "pr_number", "pr_login", "lead_days", "pr_title"]]

,category,pr_number,pr_login,lead_days,pr_title
54717,Frequent,133953,princepereira,3.332870,Fix ClusterIP load balancer disappearing when InternalTrafficPolicy: Local is set.
52919,Frequent,129672,carlory,1.414502,add a comment to features.Flexvolumes
54845,Frequent,134318,xigang,3.206146,disruption: remove unused pdb parameter from getExpectedScale method
54966,Frequent,134663,p0lyn0mial,0.476968,client-go/typed/fake: expose IsWatchListSemanticsUnSupported
54369,Occasional,133137,ivanvc,0.094572,Build etcd v3.6.3 image
52890,Occasional,129650,Adarsh-verma-14,0.296065,add missing KEP link for AuthorizeNodeWithSelectors feature gate
53486,Occasional,130859,hakuna-matatah,3.933322,Optimize DS Controller Performance: Reduce Work Duration Time & Minimize Cache Locking.
53608,Occasional,131321,karlkfi,9.095521,chore: Add VeryShortWatchError typed error
55313,Newcomer,134966,shwetha-s-poojary,48.717650,Enable notimestamp rule for Kube API Linter
53673,Newcomer,131420,simonfogliato,11.149954,Fix flake in test when run with stress on pkg/kubelet/kuberuntime TestTerminationOrder...


## Targeted sampling for outliers

Problems are rare, so following the lecture we also selected the five newcomer PRs with the longest time from first commit to merge (279 to 1,043 days), which show where the process breaks down.

In [10]:
targeted = p25[p25.category == "Newcomer"].nlargest(5, "lead_days").copy()
targeted["sample"] = "targeted"
targeted[["category", "pr_number", "pr_login", "lead_days", "pr_title"]]

,category,pr_number,pr_login,lead_days,pr_title
53410,Newcomer,128184,cupnes,1042.750949,KEP-4049: Add storage capacity scoring to VolumeBinding plugin
53459,Newcomer,129574,p-shah256,397.342025,Separate SeparateDiskTests from eviction
55403,Newcomer,129664,hkttty2009,341.910093,Fix VolumeAttachment cleanup when AttachRequired changes
54320,Newcomer,127887,soma00333,286.587870,kubelet: Support contextual logging
52907,Newcomer,124300,walker-tom,279.257187,Update EnvFromSource.Prefix doc to mention Secret as well as ConfigMap


In [11]:
# save the 17 PRs with links for reading

sample = pd.concat([stratified, targeted])
sample["url"] = "https://github.com/kubernetes/kubernetes/pull/" + sample.pr_number.astype(str)
sample = sample[["sample", "category", "pr_number", "pr_login", "lead_days", "pr_title", "url"]]
sample.to_csv("data/qual_sample.csv", index=False)
print(len(sample), "PRs to read")
sample[["sample", "category", "pr_number", "url"]]

17 PRs to read


,sample,category,pr_number,url
54717,stratified,Frequent,133953,https://github.com/kubernetes/kubernetes/pull/133953
52919,stratified,Frequent,129672,https://github.com/kubernetes/kubernetes/pull/129672
54845,stratified,Frequent,134318,https://github.com/kubernetes/kubernetes/pull/134318
54966,stratified,Frequent,134663,https://github.com/kubernetes/kubernetes/pull/134663
54369,stratified,Occasional,133137,https://github.com/kubernetes/kubernetes/pull/133137
52890,stratified,Occasional,129650,https://github.com/kubernetes/kubernetes/pull/129650
53486,stratified,Occasional,130859,https://github.com/kubernetes/kubernetes/pull/130859
53608,stratified,Occasional,131321,https://github.com/kubernetes/kubernetes/pull/131321
55313,stratified,Newcomer,134966,https://github.com/kubernetes/kubernetes/pull/134966
53673,stratified,Newcomer,131420,https://github.com/kubernetes/kubernetes/pull/131420


## Reading notes

For each PR we read the full discussion on GitHub. What we noted:

| PR | Sample | Pings by author | Bot lifecycle labels | Observation |
|---|---|---|---|---|
| [#133953](https://github.com/kubernetes/kubernetes/pull/133953) | Frequent | no | no | merged without any discussion |
| [#129672](https://github.com/kubernetes/kubernetes/pull/129672) | Frequent | no | no | one short suggestion by reviewer, author cc's reviewers himself |
| [#134318](https://github.com/kubernetes/kubernetes/pull/134318) | Frequent | no | no | only a test re-run and triage commands |
| [#134663](https://github.com/kubernetes/kubernetes/pull/134663) | Frequent | no | no | merged without any discussion |
| [#133137](https://github.com/kubernetes/kubernetes/pull/133137) | Occasional | cc's reviewers | no | discussion only about release timing, quick `/approve` |
| [#129650](https://github.com/kubernetes/kubernetes/pull/129650) | Occasional | no | no | triaged and assigned to the feature owner |
| [#130859](https://github.com/kubernetes/kubernetes/pull/130859) | Occasional | no | no | detailed review by three reviewers |
| [#131321](https://github.com/kubernetes/kubernetes/pull/131321) | Occasional | no | no | "This looks fine - but will require a rebase." |
| [#134966](https://github.com/kubernetes/kubernetes/pull/134966) | Newcomer | no | no | constructive review, author fixes quickly |
| [#131420](https://github.com/kubernetes/kubernetes/pull/131420) | Newcomer | no | no | reviewer asks "Have you run the stress?" |
| [#134022](https://github.com/kubernetes/kubernetes/pull/134022) | Newcomer | no | no | reviewer coaches the author to convert the whole test |
| [#132680](https://github.com/kubernetes/kubernetes/pull/132680) | Newcomer | no | no | "revert this doc change too please. then it's good to go." |
| [#128184](https://github.com/kubernetes/kubernetes/pull/128184) | Targeted (1,043 d) | yes | no | author asks for approval; release team code-freeze reminders |
| [#129574](https://github.com/kubernetes/kubernetes/pull/129574) | Targeted (397 d) | no | no | "please make it to the green state before it will be reviewed" while CI did not run the new test |
| [#129664](https://github.com/kubernetes/kubernetes/pull/129664) | Targeted (342 d) | yes, 3 times | rotten | "This is my first PR in Kubernetes"; maintainer had to `/remove-lifecycle rotten` |
| [#127887](https://github.com/kubernetes/kubernetes/pull/127887) | Targeted (287 d) | no | no | reviewer: "maybe @dims can tag this?", several squash requests |
| [#124300](https://github.com/kubernetes/kubernetes/pull/124300) | Targeted (279 d) | yes, same ping twice | rotten | maintainer: "/remove-lifecycle rotten the change seems to be valid" |

**Pattern:** frequent contributors' PRs pass with little or no discussion, newcomers in the stratified sample get constructive reviews, but in 3 of the 5 slowest newcomer PRs the author had to chase reviewers, and in 2 of them the bot marked the PR as rotten while it was waiting for a review. These observations are the basis for the *Potential issues* subsection.

Such a small sample shows *how* problems occur, not *how often*; the frequencies come from the metrics. Since git only contains merged PRs, abandoned PRs are missing from this sample.